In [ ]:
from google.colab import files

uploaded = files.upload()  # You will upload all 3 zip files here manually


Saving archive (3).zip to archive (3).zip
Saving archive (4).zip to archive (4).zip
Saving archive (5).zip to archive (5).zip


In [ ]:
import zipfile
import os

def unzip_local(zip_name, extract_to):
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"✅ Extracted: {zip_name} -> {extract_to}")

unzip_local("archive (3).zip", "dataset/american_sign")
unzip_local("archive (4).zip", "dataset/numbers_asl")
unzip_local("archive (5).zip", "dataset/bsl")

✅ Extracted: archive (3).zip -> dataset/american_sign
✅ Extracted: archive (4).zip -> dataset/numbers_asl
✅ Extracted: archive (5).zip -> dataset/bsl


In [ ]:
# 📦 Install and import required libraries
import shutil
import random
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
DATASET_DIRS = [
    "/content/dataset/american_sign",
    "/content/dataset/numbers_asl",
    "/content/dataset/bsl",
]
OUTPUT_DIR = "/content/all_datasets_split"
SPLIT_RATIO = 0.8

def collect_all_images(dataset_dirs):
    all_images = {}
    for dataset_path in dataset_dirs:
        for root, dirs, files in os.walk(dataset_path):
            for file in files:
                if file.lower().endswith((".jpg", ".jpeg", ".png")):
                    label = os.path.basename(root)
                    if not label.isdigit():
                        continue
                    if label not in all_images:
                        all_images[label] = []
                    all_images[label].append(os.path.join(root, file))
    return all_images

def split_and_save(all_images_dict, output_dir, split_ratio=0.8):
    for label, image_paths in all_images_dict.items():
        random.shuffle(image_paths)
        split_index = int(len(image_paths) * split_ratio)
        train_imgs = image_paths[:split_index]
        test_imgs = image_paths[split_index:]

        train_dir = os.path.join(output_dir, "train", label)
        test_dir = os.path.join(output_dir, "test", label)
        os.makedirs(train_dir, exist_ok=True)
        os.makedirs(test_dir, exist_ok=True)

        for img_path in train_imgs:
            shutil.copy(img_path, os.path.join(train_dir, os.path.basename(img_path)))
        for img_path in test_imgs:
            shutil.copy(img_path, os.path.join(test_dir, os.path.basename(img_path)))

        print(f"✅ Label {label}: {len(train_imgs)} train, {len(test_imgs)} test")

random.seed(42)
all_images = collect_all_images(DATASET_DIRS)
split_and_save(all_images, OUTPUT_DIR, SPLIT_RATIO)

import shutil
from google.colab import files

shutil.make_archive("all_datasets_split", "zip", "/content/all_datasets_split")
files.download("all_datasets_split.zip")

✅ Label 2: 1443 train, 361 test
✅ Label 1: 1443 train, 361 test
✅ Label 6: 1443 train, 361 test
✅ Label 7: 1442 train, 361 test
✅ Label 5: 1443 train, 361 test
✅ Label 4: 1443 train, 361 test
✅ Label 8: 1443 train, 361 test
✅ Label 3: 1443 train, 361 test
✅ Label 9: 1443 train, 361 test
✅ Label 0: 1443 train, 361 test


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
IMG_SIZE = 96
NUM_CLASSES = 10
BATCH_SIZE = 32
EPOCHS_INITIAL = 10
EPOCHS_FINE_TUNE = 10

# Data augmentation
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=False
)

train_gen = datagen.flow_from_directory(
    os.path.join(OUTPUT_DIR, "train"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_gen = datagen.flow_from_directory(
    os.path.join(OUTPUT_DIR, "test"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

Found 10141 images belonging to 10 classes.
Found 2865 images belonging to 10 classes.


In [ ]:
# Build MobileNetV2 model
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(optimizer=Adam(learning_rate=1e-4), loss="categorical_crossentropy", metrics=["accuracy"])

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint("best_model.h5", save_best_only=True)
]

# Phase 1: train top layers
print("\n Phase 1: Training classifier layers...")
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_INITIAL,
    callbacks=callbacks
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

🔧 Phase 1: Training classifier layers...


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 0s 335ms/step - accuracy: 0.3790 - loss: 1.8931

317/317 ━━━━━━━━━━━━━━━━━━━━ 146s 439ms/step - accuracy: 0.3795 - loss: 1.8916 - val_accuracy: 0.6778 - val_loss: 1.0200
Epoch 2/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 0s 338ms/step - accuracy: 0.7404 - loss: 0.8403

317/317 ━━━━━━━━━━━━━━━━━━━━ 138s 434ms/step - accuracy: 0.7405 - loss: 0.8402 - val_accuracy: 0.7487 - val_loss: 0.7781
Epoch 3/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 0s 337ms/step - accuracy: 0.8082 - loss: 0.6305

...

317/317 ━━━━━━━━━━━━━━━━━━━━ 139s 439ms/step - accuracy: 0.8453 - loss: 0.4992 - val_accuracy: 0.8066 - val_loss: 0.5859
Epoch 6/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step - accuracy: 0.8595 - loss: 0.4424

317/317 ━━━━━━━━━━━━━━━━━━━━ 139s 438ms/step - accuracy: 0.8595 - loss: 0.4424 - val_accuracy: 0.8220 - val_loss: 0.5532
Epoch 7/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 0s 338ms/step - accuracy: 0.8741 - loss: 0.3969

317/317 ━━━━━━━━━━━━━━━━━━━━ 138s 436ms/step - accuracy: 0.8741 - loss: 0.3969 - val_accuracy: 0.8346 - val_loss: 0.5107
Epoch 8/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 146s 462ms/step - accuracy: 0.8887 - loss: 0.3639 - val_accuracy: 0.8279 - val_loss: 0.5180
Epoch 9/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 0s 339ms/step - accuracy: 0.8913 - loss: 0.3657

317/317 ━━━━━━━━━━━━━━━━━━━━ 138s 435ms/step - accuracy: 0.8913 - loss: 0.3657 - val_accuracy: 0.8380 - val_loss: 0.4951
Epoch 10/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step - accuracy: 0.8994 - loss: 0.3205

317/317 ━━━━━━━━━━━━━━━━━━━━ 139s 438ms/step - accuracy: 0.8994 - loss: 0.3205 - val_accuracy: 0.8412 - val_loss: 0.4791


In [ ]:
# Unfreeze last 20 layers
for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(optimizer=Adam(learning_rate=1e-5), loss="categorical_crossentropy", metrics=["accuracy"])

print("\n🎯 Phase 2: Fine-tuning MobileNetV2...")
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FINE_TUNE,
    callbacks=callbacks
)


🎯 Phase 2: Fine-tuning MobileNetV2...
Epoch 1/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 189s 560ms/step - accuracy: 0.6623 - loss: 0.9865 - val_accuracy: 0.8237 - val_loss: 0.5113
Epoch 2/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 178s 561ms/step - accuracy: 0.8038 - loss: 0.5806 - val_accuracy: 0.8185 - val_loss: 0.5025
Epoch 3/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 0s 446ms/step - accuracy: 0.8379 - loss: 0.4783

317/317 ━━━━━━━━━━━━━━━━━━━━ 172s 543ms/step - accuracy: 0.8380 - loss: 0.4783 - val_accuracy: 0.8443 - val_loss: 0.4707
Epoch 4/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 0s 441ms/step - accuracy: 0.8575 - loss: 0.4192

317/317 ━━━━━━━━━━━━━━━━━━━━ 171s 538ms/step - accuracy: 0.8575 - loss: 0.4191 - val_accuracy: 0.8513 - val_loss: 0.4339
Epoch 5/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 0s 445ms/step - accuracy: 0.8725 - loss: 0.3802

317/317 ━━━━━━━━━━━━━━━━━━━━ 174s 550ms/step - accuracy: 0.8725 - loss: 0.3802 - val_accuracy: 0.8586 - val_loss: 0.4163
Epoch 6/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 0s 448ms/step - accuracy: 0.8892 - loss: 0.3389

317/317 ━━━━━━━━━━━━━━━━━━━━ 173s 545ms/step - accuracy: 0.8892 - loss: 0.3389 - val_accuracy: 0.8681 - val_loss: 0.4073
Epoch 7/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 0s 440ms/step - accuracy: 0.8991 - loss: 0.3050

317/317 ━━━━━━━━━━━━━━━━━━━━ 170s 537ms/step - accuracy: 0.8991 - loss: 0.3050 - val_accuracy: 0.8810 - val_loss: 0.3568
Epoch 8/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 0s 444ms/step - accuracy: 0.9135 - loss: 0.2649

317/317 ━━━━━━━━━━━━━━━━━━━━ 173s 545ms/step - accuracy: 0.9135 - loss: 0.2649 - val_accuracy: 0.8771 - val_loss: 0.3498
Epoch 9/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 172s 542ms/step - accuracy: 0.9173 - loss: 0.2492 - val_accuracy: 0.8866 - val_loss: 0.3508
Epoch 10/10
317/317 ━━━━━━━━━━━━━━━━━━━━ 0s 445ms/step - accuracy: 0.9181 - loss: 0.2423

317/317 ━━━━━━━━━━━━━━━━━━━━ 172s 543ms/step - accuracy: 0.9181 - loss: 0.2423 - val_accuracy: 0.8824 - val_loss: 0.3447


In [ ]:
loss, acc = model.evaluate(val_gen)
print(f"\n✅ Final Test Accuracy: {acc * 100:.2f}%")

#model.save("asl_digit_model_finetuned.h5")

model.save("asl_digit_model_finetuned.keras")

print("💾 Model saved: asl_digit_model_finetuned.h5")

90/90 ━━━━━━━━━━━━━━━━━━━━ 31s 337ms/step - accuracy: 0.8813 - loss: 0.3568

✅ Final Test Accuracy: 88.80%
💾 Model saved: asl_digit_model_finetuned.h5
